<a href="https://colab.research.google.com/github/Mohammad-Asif521/weather-app/blob/main/language_detector_through_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Step 1: Install Whisper and FFmpeg
!pip install -q git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
5

In [ ]:
# @title Step 2: Load the Language Detection Model
import whisper
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the model (this downloads about 1.5GB of weights)
print("Loading model... this may take a minute.")
model = whisper.load_model("medium", device=device)
print("✅ Model loaded successfully!")



Using device: cuda
Loading model... this may take a minute.
✅ Model loaded successfully!


In [ ]:
# @title Step 3: Define Detection Logic
def detect_language(audio_path):
    # Load audio and pad/trim it to fit 30 seconds
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)

    # Make log-Mel spectrogram and move to the same device as the model
    mel = whisper.log_mel_spectrogram(audio).to(model.device)

    # Detect the language
    _, probs = model.detect_language(mel)
    detected_lang_code = max(probs, key=probs.get)
    confidence = probs[detected_lang_code]

    # Map codes to full names for your specific requirement
    lang_map = {
        'en': 'English',
        'hi': 'Hindi'
    }

    print(f"\n🔍 Analysis for: {audio_path}")

    if detected_lang_code in lang_map:
        print(f"✅ Detected Language: **{lang_map[detected_lang_code]}**")
    else:
        print(f"⚠️ Detected Language: {detected_lang_code} (Not Hindi or English)")

    print(f"📊 Confidence Score: {confidence:.2f}")
    return detected_lang_code


In [ ]:
!pip install gradio



In [ ]:
import whisper
import gradio as gr
import torch
import numpy as np

# Detection logic for Colab (GPU) or Hugging Face (CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("turbo", device=device)

def _direct_gateway(audio_path):
    if audio_path is None: return "Waiting for Voice Input..."
    try:
        audio = whisper.load_audio(audio_path)
        audio = whisper.pad_or_trim(audio)

        # 128-bin Mel Spectrogram for Whisper-Turbo
        mel = whisper.log_mel_spectrogram(audio, n_mels=128).to(device)

        with torch.no_grad():
            _, probs = model.detect_language(mel)

        # --- THE FIX: GET THE SINGLE MAXIMUM PROBABILITY ---
        # Find the ISO code (e.g., 'hi') with the highest value
        top_iso_code = max(probs, key=probs.get)

        # Map that code to the full language name (e.g., 'Hindi')
        from whisper.tokenizer import LANGUAGES
        top_language = LANGUAGES.get(top_iso_code, top_iso_code).title()

        # Return ONLY the name of the language
        return f"Detected Language: {top_language}"

    except Exception:
        return "System Error: Please speak again."

# Professional UI: Using Textbox instead of Label for a single answer
demo = gr.Interface(
    fn=_direct_gateway,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label=" Voice Input"),
    outputs=gr.Textbox(label="RESULT", placeholder="Waiting for analysis..."),
    title="🎙️ LANGUAGE TO AUDIO DETECTOR ",
    description=" created by :Manav Soni | ",
    theme=gr.themes.Soft(primary_hue="blue")
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8252cbb397139bc7e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
